In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
import pandas as pd
from rdkit.Chem import MolFromSmiles
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split
from rdkit.Chem import AllChem, MACCSkeys, Descriptors
from mordred import Calculator, descriptors as mordred_descriptors
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from assets.functions import *
from assets.benchmark import *

df = pd.read_csv("./data/clearancehuman.csv")
df = df.rename(columns={"HLM CLint": "CLint"})

df['scaffold'] = df['smiles'].apply(compute_scaffold)

# Step 1: Get the unique scaffolds
scaffold_list = df['scaffold'].unique()

# Step 2: Split the list of scaffolds into train, test, and validation
train_scaffolds, temp_scaffolds = train_test_split(scaffold_list, test_size=0.2, random_state=42)
test_scaffolds, val_scaffolds = train_test_split(temp_scaffolds, test_size=0.5, random_state=42)

# Step 3: Create the train, test, and validation sets by filtering the original DataFrame based on scaffold
train_df = df[df['scaffold'].isin(train_scaffolds)]
test_df = df[df['scaffold'].isin(test_scaffolds)]
val_df = df[df['scaffold'].isin(val_scaffolds)]

# Step 4: Verify the distribution of scaffolds in each set
print(f"Training set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Validation set size: {len(val_df)}")

train_df["logCL"] = np.log10(train_df["CLint"].clip(lower=1e-8))
val_df["logCL"] = np.log10(val_df["CLint"].clip(lower=1e-8))
test_df["logCL"] = np.log10(test_df["CLint"].clip(lower=1e-8))

results_df = run_descriptor_benchmark(train_df, val_df, test_df, target_col="logCL")

results_df.sort_values("Test_MAE")

/home/tanush/anaconda3/envs/mordred2/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


Training set size: 2870
Test set size: 359
Validation set size: 359

🔹 Featurizing with FCFP6...


/tmp/ipykernel_1104578/3382177972.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["logCL"] = np.log10(train_df["CLint"].clip(lower=1e-8))
/tmp/ipykernel_1104578/3382177972.py:47: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df["logCL"] = np.log10(val_df["CLint"].clip(lower=1e-8))
/tmp/ipykernel_1104578/3382177972.py:48: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the

   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB

🔹 Featurizing with MACCS...
   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB

🔹 Featurizing with RDKIT...
   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB

🔹 Featurizing with MORDRED...
   ▶ Training RF
   ▶ Training SVM
   ▶ Training XGB


,Descriptor,Model,Val_MAE,Val_RMSE,Val_R2,Test_MAE,Test_RMSE,Test_R2
10,MORDRED,SVM,0.319858,0.431823,0.563993,0.333724,0.469952,0.550803
7,RDKIT,SVM,0.336711,0.453596,0.518918,0.347700,0.481754,0.527958
0,FCFP6,RF,0.334571,0.441355,0.544532,0.356155,0.495946,0.499737
11,MORDRED,XGB,0.320021,0.425768,0.576135,0.362413,0.511505,0.467856
8,RDKIT,XGB,0.327700,0.432744,0.562131,0.362628,0.505763,0.479735
6,RDKIT,RF,0.336954,0.442503,0.542160,0.370954,0.509587,0.471839
2,FCFP6,XGB,0.344917,0.454327,0.517365,0.372486,0.517737,0.454809
9,MORDRED,RF,0.338069,0.444281,0.538473,0.376541,0.515250,0.460035
5,MACCS,XGB,0.353987,0.477598,0.466656,0.387820,0.541195,0.404286
3,MACCS,RF,0.364737,0.486599,0.446364,0.391434,0.544753,0.396427
